# Requirement 24: Approved, immutable deployment

> Build and approve a known interpreter image. Reject anything else.

**Covered by:** Both

**How it is fulfilled:** The orchestrator holds the approved image ID. It compares it with the ID the sandbox actually resolved and refuses to release data on a mismatch (fail closed).

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
resolved = sandbox.get().sources_ref.disk_image.id
print("approved:", APPROVED_DISK_ID)
print("resolved:", resolved)
check("sandbox runs the approved image", resolved == APPROVED_DISK_ID)

approved: b8782cab-82d3-4eaa-880d-24549076ea98
resolved: b8782cab-82d3-4eaa-880d-24549076ea98
PASS  sandbox runs the approved image


In [3]:
# Fail closed: pretend the approved ID is different and confirm the check refuses
results = verify_session(sandbox, expected_disk_id="00000000-0000-0000-0000-000000000000")
print(results)
def release_data_if_verified(checks):
    if not all(checks.values()):
        raise PermissionError("verification failed; nothing released")
    return "released"
try:
    release_data_if_verified(results)
    check("mismatched image is refused", False)
except PermissionError as err:
    check(f"mismatched image is refused ({err})", True)
check("correct image is released", release_data_if_verified(verify_session(sandbox)) == "released")
cleanup(sandbox)

{'disk image is the approved one': False, 'created from a disk, not a snapshot': True, 'egress default is Deny': True, 'no ports exposed': True, 'suspend disabled': True, 'auto-delete enabled': True}
PASS  mismatched image is refused (verification failed; nothing released)


PASS  correct image is released


**Caveat:** This pins the platform's public image by ID. A private disk image built from Contoso's own approved container would pin their build, and is the production recommendation.